In [1]:
# load
import numpy as np
import pandas as pd
import altair as alt

demo = pd.read_csv("Demo_and_Trust_Questionaire_.csv")
tlx = pd.read_csv("Workload_Measure_TLX_.csv")
drive = pd.read_csv("Driver_performance_of_obstacle_maneuvers_.csv",
                    encoding="utf-8-sig")

print("demo ", demo.shape)
print("tlx  ", tlx.shape)
print("drive", drive.shape)

demo  (36, 28)
tlx   (33, 123)
drive (794, 14)


In [2]:
drive.head()

,subject,carMode,track,Trial,obstacle#,obstacleType,Wronglane,Crash,TTCInitTime,TTCinitDist,InitSpeed,TTCcomptime,TTCcompDist,Compspeed
0,1.0,Manual-Driving,track2,1.0,2.0,cone,0.0,0.0,3380.0,-132.954,91.714360,300.0,-11.554,85.674633
1,1.0,Manual-Driving,track2,1.0,3.0,accident,0.0,0.0,2260.0,-88.133,91.266972,840.0,-31.855,86.122021
2,1.0,Manual-Driving,track2,1.0,5.0,cone,0.0,0.0,3500.0,-150.208,99.208094,640.0,-27.466,95.069763
3,1.0,Manual-Driving,track2,1.0,6.0,accident,0.0,0.0,3780.0,-154.699,90.931432,1960.0,-81.051,91.266972
4,1.0,Manual-Driving,track1,2.0,2.0,accident,0.0,0.0,3280.0,-117.162,82.766617,1520.0,-53.508,78.740133


In [3]:
# rows vs people
print("rows        ", len(drive))
print("subjects    ", drive["subject"].nunique())
print("rows/subject", drive.groupby("subject").size().unique())
drive[["carMode", "track", "Trial", "obstacleType"]].nunique()

rows         794
subjects     33
rows/subject [24 25 23]


carMode         2
track           4
Trial           4
obstacleType    2
dtype: int64

In [4]:
pd.crosstab(drive["carMode"], drive["obstacleType"], margins=True)

obstacleType,accident,cone,All
carMode,,,
Manual-Driving,263,265,528
Self-Driving,132,132,264
All,395,397,792


In [5]:
# what's missing
drive = drive.dropna(subset=["subject"])
drive.isna().sum()

subject          0
carMode          0
track            0
Trial            0
obstacle#        0
obstacleType     0
Wronglane        0
Crash            0
TTCInitTime     93
TTCinitDist     93
InitSpeed       93
TTCcomptime     93
TTCcompDist     97
Compspeed       97
dtype: int64

In [6]:
# blank TTC vs wrong-lane
pd.crosstab(drive["Wronglane"], drive["TTCInitTime"].isna(), margins=True)

TTCInitTime,False,True,All
Wronglane,,,
0.0,699,9,708
1.0,0,84,84
All,699,93,792


In [7]:
# wrong-lane rate by mode and obstacle
drive["mode"] = drive["carMode"].map({"Manual-Driving": "Manual",
                                      "Self-Driving": "Takeover"})
drive.groupby(["mode", "obstacleType"])["Wronglane"].agg(
    ["size", "sum", "mean"]).round(3)

size   sum   mean
mode     obstacleType                   
Manual   accident       263  28.0  0.106
         cone           265  22.0  0.083
Takeover accident       132  33.0  0.250
         cone           132   1.0  0.008

In [8]:
# questionnaire: 16 items, 6-point scale
items = [str(i) for i in range(1, 17)]
demo[items].describe().loc[["count", "mean", "min", "max"]].round(2)

,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
count,34.00,34.00,34.00,34.00,34.00,34.00,34.00,32.00,32.00,32.00,32.00,32.00,32.00,32.00,32.00,32.00
mean,4.26,3.53,3.68,3.35,2.74,3.53,4.88,5.03,3.66,4.22,3.81,3.09,3.38,3.72,4.56,3.12
min,2.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,2.00,1.00
max,6.00,6.00,6.00,6.00,5.00,6.00,6.00,6.00,6.00,6.00,6.00,6.00,6.00,6.00,6.00,6.00


In [9]:
# average items only when all were answered
trust_items = ["2", "3", "4", "5"]
views_items = ["9", "10", "11", "12"]

demo["trust"] = demo[trust_items].mean(axis=1).where(
    demo[trust_items].notna().all(axis=1))
demo["views"] = demo[views_items].mean(axis=1).where(
    demo[views_items].notna().all(axis=1))

demo[["trust", "views"]].describe().round(2)

,trust,views
count,34.00,32.00
mean,3.32,3.70
std,1.16,1.26
min,1.50,1.00
25%,2.31,2.75
50%,3.12,3.75
75%,4.25,4.75
max,5.25,6.00


In [10]:
# TLX cols 1-8: which runs were manual (m) or automated (a)
tlx.iloc[:6, :9]

,Subject,1,2,3,4,5,6,7,8
0,1,m,a,m,a,a,m,a,m
1,2,a,m,a,m,m,a,m,a
2,3,m,a,m,a,a,m,a,m
3,4,a,m,a,m,m,a,m,a
4,5,a,m,a,m,m,a,m,a
5,6,m,a,m,a,a,m,a,m


In [11]:
# are cols 25/26 the manual and automated means?
mode_cols = [str(i) for i in range(1, 9)]
run_cols = [str(i) for i in range(17, 25)]

letters = tlx[mode_cols].apply(lambda s: s.str.strip().str.lower())
runs = tlx[run_cols]

man = runs.where(letters.values == "m").mean(axis=1)
aut = runs.where(letters.values == "a").mean(axis=1)

print((man - tlx["25"]).abs().max())
print((aut - tlx["26"]).abs().max())

7.499995291482264e-09
7.5000059496233e-09


In [12]:
work = tlx[["Subject", "25", "26"]].copy()
work.columns = ["subject", "tlx_manual", "tlx_auto"]
work.describe().round(1)

,subject,tlx_manual,tlx_auto
count,33.0,33.0,33.0
mean,19.2,33.2,27.9
std,11.2,20.2,13.4
min,1.0,6.5,5.8
25%,10.0,19.2,18.8
50%,20.0,27.1,25.2
75%,29.0,51.9,42.1
max,37.0,90.7,56.0


In [13]:
# per-subject behaviour by mode
beh = drive.pivot_table(index="subject", columns="mode",
                        values=["InitSpeed", "Wronglane"], aggfunc="mean")
beh.columns = [f"{v.lower()}_{m.lower()}" for v, m in beh.columns]
beh = beh.reset_index()
beh["subject"] = beh["subject"].astype(int)
beh.describe().round(2)

,subject,initspeed_manual,initspeed_takeover,wronglane_manual,wronglane_takeover
count,33.00,33.00,33.00,33.00,33.00
mean,19.18,92.55,75.91,0.10,0.13
std,11.24,8.41,3.22,0.21,0.02
min,1.00,71.78,71.55,0.00,0.12
25%,10.00,91.50,73.96,0.00,0.12
50%,20.00,95.73,74.60,0.00,0.12
75%,29.00,99.82,77.16,0.06,0.12
max,37.00,100.07,87.78,0.75,0.25


In [14]:
# one row per subject
sub = (demo[["Subject", "trust", "views"]]
       .rename(columns={"Subject": "subject"})
       .merge(work, on="subject")
       .merge(beh, on="subject"))

print(sub.shape)
sub.head()

(33, 9)


,subject,trust,views,tlx_manual,tlx_auto,initspeed_manual,initspeed_takeover,wronglane_manual,wronglane_takeover
0,1,5.00,5.00,52.546296,32.407407,91.567561,73.483334,0.000000,0.125
1,2,2.00,3.00,60.185185,42.361111,79.560343,73.547247,0.625000,0.125
2,3,3.75,4.75,17.129630,19.907407,99.927109,79.618929,0.125000,0.125
3,4,2.25,4.75,23.842593,22.916667,92.042910,75.688314,0.000000,0.125
4,5,3.25,2.00,28.240741,42.824074,81.700078,73.994634,0.117647,0.125


In [15]:
# does trust relate to anything?
pairs = [("trust", "tlx_auto"),
         ("trust", "tlx_manual"),
         ("views", "initspeed_manual"),
         ("trust", "wronglane_manual"),
         ("trust", "wronglane_takeover")]

for x, y in pairs:
    d = sub[[x, y]].dropna()
    print(f"{x:6s} vs {y:19s} r = {d[x].corr(d[y]):+.3f}   n = {len(d)}")

trust  vs tlx_auto            r = -0.502   n = 33
trust  vs tlx_manual          r = -0.293   n = 33
views  vs initspeed_manual    r = +0.489   n = 31
trust  vs wronglane_manual    r = -0.349   n = 33
trust  vs wronglane_takeover  r = +0.140   n = 33


In [16]:
# report gives r = .39 .53 .36 .44 for items 9-12 vs manual speed
chk = (demo[["Subject"] + views_items]
       .rename(columns={"Subject": "subject"})
       .merge(beh[["subject", "initspeed_manual"]], on="subject"))

for it in views_items:
    d = chk[[it, "initspeed_manual"]].dropna()
    print(f"item {it:>2}   r = {d[it].corr(d['initspeed_manual']):+.3f}")

item  9   r = +0.383
item 10   r = +0.495
item 11   r = +0.355
item 12   r = +0.431


In [17]:
# are the two scales measuring the same thing?
d = sub[["trust", "views"]].dropna()
r = d["trust"].corr(d["views"])
print(f"r = {r:+.3f}   r^2 = {r**2:.3f}   n = {len(d)}")

r = +0.285   r^2 = 0.081   n = 31


In [18]:
# each scale against each outcome
cells = []
for pred in ["trust", "views"]:
    for col, lab in [("tlx_auto", "Workload when handed control"),
                     ("initspeed_manual", "Speed chosen when driving")]:
        d = sub[[pred, col]].dropna()
        cells.append({"predictor": pred, "outcome": lab,
                      "r": d[pred].corr(d[col]), "n": len(d)})

cells = pd.DataFrame(cells)
cells.round(3)

,predictor,outcome,r,n
0,trust,Workload when handed control,-0.502,33
1,trust,Speed chosen when driving,0.096,33
2,views,Workload when handed control,-0.345,31
3,views,Speed chosen when driving,0.489,31


In [19]:
# palette, fixed across every chart
BLUE, ORANGE, RED = "#2a78d6", "#eb6834", "#e34948"
INK, MUTED, GRID = "#0b0b0b", "#898781", "#e1e0d9"

wl = sub.melt(id_vars=["subject", "trust"],
              value_vars=["tlx_auto", "tlx_manual"],
              var_name="condition", value_name="workload")
wl["condition"] = wl["condition"].map({"tlx_auto": "Take-over",
                                       "tlx_manual": "Manual"})
wl.head()

,subject,trust,condition,workload
0,1,5.00,Take-over,32.407407
1,2,2.00,Take-over,42.361111
2,3,3.75,Take-over,19.907407
3,4,2.25,Take-over,22.916667
4,5,3.25,Take-over,42.824074


In [20]:
# first attempt: does trust matter more when control is handed back?
base = alt.Chart().encode(
    x=alt.X("trust:Q", title="Trust (items 2-5)", scale=alt.Scale(domain=[1, 6])),
    y=alt.Y("workload:Q", title="NASA-TLX workload", scale=alt.Scale(domain=[0, 100])),
    color=alt.Color("condition:N",
                    scale=alt.Scale(domain=["Take-over", "Manual"],
                                    range=[ORANGE, BLUE]), legend=None),
)

alt.layer(
    base.mark_circle(size=80, opacity=0.7),
    base.transform_regression("trust", "workload",
                              groupby=["condition"]).mark_line(size=2.5),
    data=wl,
).properties(width=250, height=250).facet(
    column=alt.Column("condition:N", title=None, sort=["Take-over", "Manual"]))

alt.FacetChart(...)

In [21]:
for c in ["Take-over", "Manual"]:
    d = wl[wl["condition"] == c].dropna(subset=["trust", "workload"])
    b, a = np.polyfit(d["trust"], d["workload"], 1)
    resid = d["workload"] - (a + b * d["trust"])
    print(f"{c:10s} slope = {b:+6.2f}   residual SD = {resid.std():5.1f}"
          f"   r = {d['trust'].corr(d['workload']):+.2f}")

Take-over  slope =  -5.75   residual SD =  11.6   r = -0.50
Manual     slope =  -5.05   residual SD =  19.4   r = -0.29


In [22]:
# keep the take-over panel only
tk = wl[wl["condition"] == "Take-over"]
r1 = tk["trust"].corr(tk["workload"])

base1 = alt.Chart().encode(
    x=alt.X("trust:Q", title="Trust in the automation (items 2-5)",
            scale=alt.Scale(domain=[1, 6])),
    y=alt.Y("workload:Q", title="NASA-TLX workload (0-100)",
            scale=alt.Scale(domain=[0, 100])),
)

chart1 = alt.layer(
    base1.mark_circle(size=80, opacity=0.7, color=ORANGE),
    base1.transform_regression("trust", "workload").mark_line(size=2.5, color=ORANGE),
    data=tk,
).properties(
    width=250, height=250,
    title=alt.Title("Drivers who trust the system find the hand-off easier",
                    subtitle=f"r = {r1:+.2f}   n = {len(tk)}",
                    anchor="start", fontSize=13, subtitleFontSize=11,
                    subtitleColor=MUTED))

chart1

alt.LayerChart(...)

In [23]:
# does enthusiasm show up in how fast people drive themselves?
sp = sub.dropna(subset=["views", "initspeed_manual"])
r2 = sp["views"].corr(sp["initspeed_manual"])

base2 = alt.Chart().encode(
    x=alt.X("views:Q", title="Views of automation (items 9-12)",
            scale=alt.Scale(domain=[1, 6])),
    y=alt.Y("initspeed_manual:Q", title="Initial speed, manual driving",
            scale=alt.Scale(domain=[70, 102])),
)

chart2 = alt.layer(
    base2.mark_circle(size=80, opacity=0.7, color=BLUE),
    base2.transform_regression("views", "initspeed_manual").mark_line(
        size=2.5, color=BLUE),
    data=sp,
).properties(
    width=250, height=250,
    title=alt.Title("Enthusiasts drive faster when they drive themselves",
                    subtitle=f"r = {r2:+.2f}   n = {len(sp)}",
                    anchor="start", fontSize=13, subtitleFontSize=11,
                    subtitleColor=MUTED))

chart2

alt.LayerChart(...)

In [24]:
# that row of points at 100 - is it a ceiling?
man = drive.loc[drive["mode"] == "Manual", "InitSpeed"].dropna()
print("max speed in the data", round(man.max(), 2))
print("maneuvers above 99.5 ", (man > 99.5).sum(), "of", len(man))

max speed in the data 100.1
maneuvers above 99.5  232 of 470


In [25]:
# What does performance look like
print(sub["wronglane_manual"].value_counts().sort_index().to_string())
print()
print(sub["wronglane_takeover"].value_counts().sort_index().to_string())

wronglane_manual
0.000000    21
0.062500     5
0.117647     1
0.125000     2
0.375000     1
0.625000     1
0.733333     1
0.750000     1

wronglane_takeover
0.125    32
0.250     1


In [26]:
# How many drivers is that resting on?
d = sub[["trust", "wronglane_manual"]].dropna()
hi = d.nlargest(4, "wronglane_manual")
d2 = d.drop(hi.index)

print(f"all {len(d)}            r   = {d['trust'].corr(d['wronglane_manual']):+.3f}")
print(f"spearman            rho = {d['trust'].corr(d['wronglane_manual'], method='spearman'):+.3f}")
print(f"minus top 4 (n={len(d2)})   r   = {d2['trust'].corr(d2['wronglane_manual']):+.3f}")
hi

all 33            r   = -0.349
spearman            rho = -0.288
minus top 4 (n=29)   r   = -0.113


,trust,wronglane_manual
11,1.50,0.750000
5,2.25,0.733333
1,2.00,0.625000
15,4.25,0.375000


In [27]:
# so: a distribution, not a relationship
wr = sub.melt(id_vars=["subject"],
              value_vars=["wronglane_manual", "wronglane_takeover"],
              var_name="condition", value_name="rate")
wr["condition"] = wr["condition"].map({"wronglane_manual": "Manual",
                                       "wronglane_takeover": "Take-over"})

chart3 = alt.Chart(wr).mark_bar().encode(
    x=alt.X("rate:Q", bin=alt.Bin(step=0.05, extent=[0, 0.8]),
            title="Share of obstacles taken in the wrong lane"),
    y=alt.Y("count():Q", title="Drivers", axis=alt.Axis(format="d")),
    color=alt.Color("condition:N",
                    scale=alt.Scale(domain=["Take-over", "Manual"],
                                    range=[ORANGE, BLUE]), legend=None),
).properties(width=250, height=130).facet(
    row=alt.Row("condition:N", title=None, sort=["Manual", "Take-over"],
                header=alt.Header(labelOrient="top", labelAngle=0,
                                  labelAnchor="start", labelFontSize=12,
                                  labelFontWeight=600, labelColor=INK)),
).properties(
    title=alt.Title("Automation erased the differences between drivers",
                    subtitle="Each bar counts drivers; n = 33",
                    anchor="start", fontSize=13, subtitleFontSize=11,
                    subtitleColor=MUTED))

chart3

alt.FacetChart(...)

In [28]:
# do the 16 items group the way the report says?
q33 = demo[demo["Subject"].isin(sub["subject"])]
cm = q33[items].corr()
cm.round(2)

,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
1,1.00,0.12,0.12,-0.05,0.00,0.24,-0.08,0.28,-0.19,-0.41,-0.26,-0.31,-0.16,0.33,0.17,-0.42
2,0.12,1.00,0.93,0.81,0.42,-0.33,0.15,0.18,-0.01,0.02,0.09,0.08,0.07,0.13,0.07,0.01
3,0.12,0.93,1.00,0.80,0.46,-0.40,0.12,0.23,0.08,0.08,0.16,0.06,0.12,0.07,0.12,0.02
4,-0.05,0.81,0.80,1.00,0.64,-0.52,0.22,0.21,0.21,0.19,0.33,0.28,0.12,-0.22,0.05,0.27
5,0.00,0.42,0.46,0.64,1.00,-0.55,0.36,-0.16,0.52,0.41,0.50,0.41,0.44,-0.29,-0.28,0.55
6,0.24,-0.33,-0.40,-0.52,-0.55,1.00,-0.17,0.11,-0.27,-0.34,-0.41,-0.40,-0.13,0.36,0.09,-0.40
7,-0.08,0.15,0.12,0.22,0.36,-0.17,1.00,-0.21,0.49,0.24,0.65,0.44,0.18,-0.31,-0.20,0.49
8,0.28,0.18,0.23,0.21,-0.16,0.11,-0.21,1.00,-0.22,-0.21,-0.20,-0.48,-0.43,0.00,0.03,-0.29
9,-0.19,-0.01,0.08,0.21,0.52,-0.27,0.49,-0.22,1.00,0.74,0.71,0.70,0.59,-0.43,-0.36,0.78
10,-0.41,0.02,0.08,0.19,0.41,-0.34,0.24,-0.21,0.74,1.00,0.59,0.56,0.62,-0.52,-0.16,0.65


In [29]:
# Correlations decide the order
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform

dist = 1 - cm.values
np.fill_diagonal(dist, 0)
order = [items[i] for i in
         leaves_list(linkage(squareform(dist, checks=False), method="average"))]
print(order)

['5', '4', '2', '3', '7', '13', '12', '10', '11', '9', '16', '15', '8', '1', '6', '14']


In [30]:
# heatmap, ordered by cluster, diagonal dropped
cmm = cm.mask(np.eye(16, dtype=bool))
cmm.index.name = "item_y"
hm = (cmm.reset_index()
         .melt(id_vars="item_y", var_name="item_x", value_name="r")
         .dropna())

chart4 = alt.Chart(hm).mark_rect().encode(
    x=alt.X("item_x:O", sort=order, title="Questionnaire item"),
    y=alt.Y("item_y:O", sort=order, title=None),
    color=alt.Color("r:Q",
                    scale=alt.Scale(domain=[-1, 0, 1],
                                    range=[BLUE, "#f0efec", RED],
                                    interpolate="lab"),
                    legend=alt.Legend(title="r", direction="horizontal",
                                      orient="bottom", gradientLength=120)),
).properties(
    width=300, height=300,
    title=alt.Title("The questionnaire splits into two blocks",
                    subtitle="Items 2-5 = trust in the system; 9-13, 16 = views of automation; n = 33",
                    anchor="start", fontSize=13, subtitleFontSize=11,
                    subtitleColor=MUTED))

chart4

alt.Chart(...)

In [31]:
# each scale predicts its own outcome
chart5 = alt.layer(
    alt.Chart().mark_rule(color=MUTED, size=1).encode(x=alt.datum(0)),
    alt.Chart().mark_bar(height=16, color=INK, opacity=0.85).encode(
        x=alt.X("r:Q", title="Correlation with the outcome",
                scale=alt.Scale(domain=[-0.6, 0.6])),
        y=alt.Y("predictor:N", title=None, sort=["trust", "views"])),
    data=cells,
).properties(width=440, height=110).facet(
    row=alt.Row("outcome:N", title=None,
                sort=["Workload when handed control",
                      "Speed chosen when driving"],
                header=alt.Header(labelOrient="top", labelAngle=0,
                                  labelAnchor="start", labelFontSize=12,
                                  labelFontWeight=600, labelColor=INK)),
).properties(
    title=alt.Title("Trust and enthusiasm predict different things",
                    subtitle="r = +0.29 between the two scales, so they are not the same measure",
                    anchor="start", fontSize=13, subtitleFontSize=11,
                    subtitleColor=MUTED))

chart5

alt.FacetChart(...)

In [32]:
poster = alt.vconcat(
    alt.hconcat(chart1, chart2, chart3, spacing=34),
    alt.hconcat(chart4, chart5, spacing=34),
    spacing=24
).properties(
    title=alt.Title(
        "Do people who trust automation drive differently?",
        subtitle=[
            "Trust changes how the drive feels and how fast drivers choose to go - but not how well they avoid obstacles.",
            "Trust = items 2-5 (dependable / reliable / can trust).   Views = items 9-12 (enjoyable / easier / frees me up).   Orange = control handed back, blue = manual.",
            "Driving simulator, 33 drivers, 8 tracks, 792 obstacle encounters.   Yerim Mo",
            "Data: Miles & Strybel (2020), Harvard Dataverse.  https://doi.org/10.7910/DVN/BWCCET",
        ],
        anchor="start", fontSize=20, subtitleFontSize=11.5,
        subtitleColor=MUTED, offset=10)
).configure_view(
    stroke=None
).configure_axis(
    grid=True, gridColor=GRID, gridWidth=0.8,
    domainColor="#c3c2b7", tickColor="#c3c2b7",
    labelColor="#52514e", titleColor=INK,
    labelFontSize=10.5, titleFontSize=11
).configure_legend(
    labelColor="#52514e", titleColor=INK
).configure_title(
    color=INK
).configure(
    background="#ffffff"
)

poster

alt.VConcatChart(...)

In [33]:
poster.save("Yerim Mo_Data Visualization_Final Project_v2.pdf")